In [1]:
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
# ---------------- داده‌ها ------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

In [ ]:
# -------- آماده‌سازی داده برای Inception --------
def prepare_incep_dataset(x, y, batch_size=32):
    def preprocess(img, label):
        img = tf.image.resize(img, (299, 299))
        img = preprocess_input(img)
        return img, label
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.map(preprocess).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds_incep = prepare_incep_dataset(x_train, y_train_cat)
test_ds_incep = prepare_incep_dataset(x_test, y_test_cat)

In [ ]:
# -------- مدل InceptionV3 --------
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
base_model.trainable = False  # می‌تونی True بذاری برای Fine-tune

model_incep = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model_incep.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
print("\n🔧 در حال آموزش مدل InceptionV3...")
model_incep.fit(train_ds_incep, epochs=5, verbose=2)
test_loss_incep, test_acc_incep = model_incep.evaluate(test_ds_incep, verbose=0)
print(f"\n✅ دقت InceptionV3: {test_acc_incep:.4f}")

In [ ]:
# پیش‌بینی و ارزیابی
y_pred_incep = model_incep.predict(test_ds_incep)
y_pred_classes_incep = np.argmax(y_pred_incep, axis=1)

print("\n📊 گزارش طبقه‌بندی InceptionV3:")
print(classification_report(y_test, y_pred_classes_incep))

conf_mat_incep = confusion_matrix(y_test, y_pred_classes_incep)
plt.figure(figsize=(10, 8))
sns.heatmap(conf_mat_incep, annot=True, fmt='d', cmap='Blues')
plt.title("ماتریس درهم‌ریختگی - InceptionV3")
plt.xlabel("پیش‌بینی‌شده")
plt.ylabel("واقعی")
plt.show()

In [ ]:
# -------- مدل CNN ساده --------
x_train_small = x_train / 255.0
x_test_small = x_test / 255.0

model_cnn = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
print("\n🔧 در حال آموزش مدل CNN ساده...")
model_cnn.fit(x_train_small, y_train_cat, epochs=5, batch_size=32, validation_split=0.1, verbose=2)
test_loss_cnn, test_acc_cnn = model_cnn.evaluate(x_test_small, y_test_cat, verbose=0)
print(f"\n✅ دقت CNN ساده: {test_acc_cnn:.4f}")

In [ ]:
# پیش‌بینی و ارزیابی
y_pred_cnn = model_cnn.predict(x_test_small)
y_pred_classes_cnn = np.argmax(y_pred_cnn, axis=1)

print("\n📊 گزارش طبقه‌بندی CNN ساده:")
print(classification_report(y_test, y_pred_classes_cnn))

conf_mat_cnn = confusion_matrix(y_test, y_pred_classes_cnn)
plt.figure(figsize=(10, 8))
sns.heatmap(conf_mat_cnn, annot=True, fmt='d', cmap='Greens')
plt.title("ماتریس درهم‌ریختگی - CNN ساده")
plt.xlabel("پیش‌بینی‌شده")
plt.ylabel("واقعی")
plt.show()

In [ ]:
# -------- مقایسه نهایی --------
print("\n📈 مقایسه دقت دو مدل:")
print(f"✔️ InceptionV3 Accuracy: {test_acc_incep:.4f}")
print(f"✔️ Simple CNN Accuracy : {test_acc_cnn:.4f}")